# Generate sample reconstructions for a selected sweep model

This notebook mirrors the reconstruction flow used in the existing 2025 sweep notebooks, but keeps everything inline (no wrapper call and no image-file export).

It explicitly shows: run discovery, config/checkpoint loading, model rehydration, prediction, and plotting original vs reconstructed images for inspection.

In [ ]:
import os
import sys
import pickle
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import pytorch_lightning as pl

# Ensure repository imports resolve when running notebook from results/...
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "morphseq" and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.core.run.run_utils import initialize_model
from src.core.lightning.pl_wrappers import LitModel
from src.core.data.dataset_configs import BaseDataConfig

In [ ]:
# ===== User selections =====
DATA_ROOT = Path("/net/trapnell/vol1/home/nlammers/projects/data/morphseq/training_data/models")
SWEEP_NAME = "sweep10"
SELECTED_RUN = None  # set to a full run folder name to force a specific run
RUN_INDEX = -1       # if SELECTED_RUN is None, pick this index from sorted runs
N_SAMPLES = 24
RANDOM_SEED = 42
BATCH_SIZE = N_SAMPLES

# Discover candidate runs
training_outputs = DATA_ROOT / "training_outputs"
candidate_runs = sorted([p for p in training_outputs.glob(f"{SWEEP_NAME}_*") if p.is_dir()])
assert len(candidate_runs) > 0, f"No runs found for pattern: {SWEEP_NAME}_*"

if SELECTED_RUN is not None:
    run_path = training_outputs / SELECTED_RUN
    assert run_path.exists(), f"Selected run does not exist: {run_path}"
else:
    run_path = candidate_runs[RUN_INDEX]

cfg_path = run_path / ".hydra" / "config.yaml"
ckpt_path = run_path / "checkpoints" / "last.ckpt"
split_path = run_path / "split_indices.pkl"

if not ckpt_path.exists():
    ckpts = sorted((run_path / "checkpoints").glob("*.ckpt"))
    assert len(ckpts) > 0, f"No checkpoints found in {(run_path / 'checkpoints')}"
    ckpt_path = ckpts[-1]

print(f"Using run: {run_path.name}")
print(f"Config: {cfg_path}")
print(f"Checkpoint: {ckpt_path}")
print(f"Split file: {split_path}")

In [ ]:
# ===== Load config/model/checkpoint (explicitly, without recon wrapper) =====
config = OmegaConf.load(str(cfg_path))
config_dict = OmegaConf.to_container(config, resolve=True)

model, model_config, train_data_config, loss_fn, _, train_config = initialize_model(config_dict)

with open(split_path, "rb") as f:
    split_dict = pickle.load(f)

eval_data_config = BaseDataConfig(
    train_indices=np.asarray(split_dict["train"]),
    test_indices=np.asarray(split_dict["test"]),
    eval_indices=np.asarray(split_dict["eval"]),
    root=train_data_config.root,
    return_sample_names=True,
    transform_name="basic",
)
eval_data_config.make_metadata()

lit_model = LitModel.load_from_checkpoint(
    str(ckpt_path),
    model=model,
    loss_fn=loss_fn,
    data_cfg=eval_data_config,
    train_cfg=train_config,
)
lit_model.eval().freeze()
print("Model loaded and frozen.")

In [ ]:
# ===== Sample test images and run prediction =====
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

dataset = eval_data_config.create_dataset()
test_indices = np.asarray(eval_data_config.test_indices)
sampled = np.random.permutation(test_indices)[:N_SAMPLES]
sampler = SubsetRandomSampler(sampled)

dl = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    num_workers=eval_data_config.num_workers,
    sampler=sampler,
    shuffle=False,
)

trainer = pl.Trainer(accelerator="auto", devices=1, limit_predict_batches=1)
lit_model.current_mode = "test"
preds = trainer.predict(lit_model, dataloaders=dl)

assert len(preds) > 0, "No predictions returned."
batch_pred = preds[0]
orig = batch_pred["orig"].detach().cpu()
recon = batch_pred["recon"].detach().cpu()
snip_ids = batch_pred["snip_ids"]
recon_loss = batch_pred["recon_loss"].detach().cpu().numpy()

print(f"Predicted {orig.shape[0]} samples.")

In [ ]:
# ===== Display originals and reconstructions in a single figure =====
n_show = orig.shape[0]
fig, axes = plt.subplots(n_show, 2, figsize=(8, max(2*n_show, 6)), squeeze=False)

for i in range(n_show):
    x = orig[i].squeeze().numpy()
    y = recon[i].squeeze().numpy()
    name = Path(snip_ids[i]).stem

    axes[i, 0].imshow(x, cmap="gray")
    axes[i, 0].set_title(f"orig | {name}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(y, cmap="gray")
    axes[i, 1].set_title(f"recon | loss={recon_loss[i]:.1f}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()